In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install numpy evaluate datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import evaluate
from datasets import load_dataset, Audio
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# STAGE 1: Configuration & Setup
# ==========================================
# Define paths and model constants
DATA_DIR = "/content/drive/MyDrive/Datasets/Rakshai_dataset_augmented" # Ensure 'real/' and 'fake/' are inside this directory
MODEL_ID = "facebook/wav2vec2-base"
SAMPLING_RATE = 16000 # Wav2Vec2 strictly requires 16kHz
BATCH_SIZE = 8
EPOCHS = 5

# ==========================================
# STAGE 2: Dataset Preparation
# ==========================================
def prepare_dataset(data_dir):
    """
    Loads the dataset using Hugging Face's audiofolder script.
    Automatically infers labels from the 'real' and 'fake' subdirectories.
    """
    print("Loading dataset from folders...")
    # The audiofolder feature maps subdirectories to label classes
    dataset = load_dataset("audiofolder", data_dir=data_dir, drop_labels=False)

    # Split the dataset into train and validation sets (80/20 split)
    dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

    # Cast the audio column to the required 16kHz sampling rate
    dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

    return dataset

In [4]:
# ==========================================
# STAGE 3: Preprocessing
# ==========================================
def preprocess_function(examples, feature_extractor):
    """
    Extracts features from the raw audio arrays.
    Pads or truncates the audio to a uniform length (e.g., 4 seconds) to handle memory effectively.
    """
    audio_arrays = [x["array"] for x in examples["audio"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLING_RATE,
        padding=True,
        max_length=SAMPLING_RATE * 4, # 4 seconds max
        truncation=True,
        return_tensors="pt"
    )

    return inputs

In [5]:
# ==========================================
# STAGE 4: Model Initialization
# ==========================================
def initialize_model(label2id, id2label):
    """
    Loads the pre-trained feature extractor and sequence classification model.
    """
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    )

    # FIX: Updated method name from freeze_feature_extractor() to freeze_feature_encoder()
    model.freeze_feature_encoder()

    return feature_extractor, model

In [6]:
# ==========================================
# STAGE 5: Metrics & Training Execution
# ==========================================
def compute_metrics(eval_pred):
    """Computes accuracy and F1 score during evaluation."""
    metric_acc = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")

    predictions = np.argmax(eval_pred.predictions, axis=1)

    accuracy = metric_acc.compute(predictions=predictions, references=eval_pred.label_ids)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=eval_pred.label_ids)["f1"]

    return {"accuracy": accuracy, "f1": f1}

def train_model():
    # 1. Load Data
    dataset = prepare_dataset(DATA_DIR)

    # Extract label mappings dynamically
    labels = dataset["train"].features["label"].names
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for i, label in enumerate(labels)}

    # 2. Initialize Model & Extractor
    feature_extractor, model = initialize_model(label2id, id2label)

    # 3. Apply Preprocessing
    print("Extracting features (this may take a while)...")
    encoded_dataset = dataset.map(
        lambda x: preprocess_function(x, feature_extractor),
        batched=True,
        batch_size=BATCH_SIZE,
        remove_columns=["audio"] # Remove raw audio to save memory
    )

# 4. Define Training Arguments
    training_args = TrainingArguments(
        output_dir="./deepfake_audio_model",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        warmup_steps=50,  # FIX 1: Changed from warmup_ratio to avoid future deprecation
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=True,
    )

    # 5. Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["test"],
        processing_class=feature_extractor, # FIX 2: Changed from tokenizer to processing_class
        compute_metrics=compute_metrics,
    )

    # 6. Train and Save
    print("Starting training...")
    trainer.train()

    print("Evaluating model...")
    results = trainer.evaluate()
    print(f"Final Results: {results}")

    trainer.save_model("./best_deepfake_model")
    print("Model saved successfully to ./best_deepfake_model")

if __name__ == "__main__":
    train_model()

Loading dataset from folders...


Resolving data files:   0%|          | 0/5184 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating model...


Final Results: {'eval_loss': 0.007691513746976852, 'eval_accuracy': 0.9990356798457087, 'eval_f1': 0.9990356798457087, 'eval_runtime': 47.022, 'eval_samples_per_second': 22.054, 'eval_steps_per_second': 2.765, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully to ./best_deepfake_model


In [11]:
import torch
import librosa
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_PATH = "/content/best_deepfake_model" # The folder we saved to in Stage 6
TARGET_SAMPLING_RATE = 16000 # Wav2Vec 2.0 strictly requires 16kHz

# ==========================================
# LOAD PIPELINE
# ==========================================
def load_pipeline(model_path):
    """Loads the saved feature extractor and the trained model."""
    print("Loading model and feature extractor...")
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_path)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(model_path)

    # Move model to GPU if available for faster inference
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval() # Set model to evaluation mode (turns off dropout, etc.)

    return feature_extractor, model, device

# ==========================================
# PREDICTION FUNCTION
# ==========================================
def predict_audio(audio_path, feature_extractor, model, device):
    """
    Loads an audio file, preprocesses it, and runs it through the model
    to predict if it is 'real' or 'fake'.
    """
    # 1. Load and Resample Audio
    # librosa automatically resamples the audio to 16kHz if it isn't already
    print(f"Processing {audio_path}...")
    speech_array, _ = librosa.load(audio_path, sr=TARGET_SAMPLING_RATE)

    # 2. Extract Features
    inputs = feature_extractor(
        speech_array,
        sampling_rate=TARGET_SAMPLING_RATE,
        return_tensors="pt",
        padding=True
    )

    # Move inputs to the same device as the model
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # 3. Model Inference
    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = model(**inputs)
        logits = outputs.logits

    # 4. Interpret Results
    # Apply softmax to get percentages (probabilities)
    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    # Get the highest probability class
    predicted_id = torch.argmax(probabilities, dim=-1).item()
    predicted_label = model.config.id2label[predicted_id]
    confidence_score = probabilities[0][predicted_id].item() * 100

    return predicted_label, confidence_score

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # Point this to a test file (ensure it was NOT in your training set)
    test_audio_file = "/content/Recording (3).m4a"

    # Initialize
    extractor, model, compute_device = load_pipeline(MODEL_PATH)

    # Predict
    try:
        label, confidence = predict_audio(test_audio_file, extractor, model, compute_device)
        print("\n" + "="*30)
        print(f"PREDICTION: {label.upper()}")
        print(f"CONFIDENCE: {confidence:.2f}%")
        print("="*30 + "\n")
    except Exception as e:
        print(f"An error occurred during prediction: {e}")

Loading model and feature extractor...


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Processing /content/Recording (3).m4a...


/tmp/ipykernel_1442/3971521635.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech_array, _ = librosa.load(audio_path, sr=TARGET_SAMPLING_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



PREDICTION: FAKE
CONFIDENCE: 99.82%

